In [1]:
import os
model_to_test = 'baseline'
model_revision = 'baseline'
hls4ml_revision = 'VU'

base_dir = os.path.abspath(model_to_test)
model_dir = os.path.join(base_dir, model_revision)
os.makedirs(model_dir, exist_ok=True)

description = f"""
# Model Configuration

- **Model architecture description**: {model_to_test}
- **Model Revision**: {model_revision}
- **HLS4ML Revision**: {hls4ml_revision}
- **Target Device**: KV260 (xck26-sfvc784-2LV-c)
- **Dataset**: HLS4ML LHC Jets
- **Vivado/Vitis**: 2025.2
"""
output_dir = os.path.join(model_dir, f"hls4ml_prj_{hls4ml_revision}")
os.makedirs(output_dir, exist_ok=True)
with open(os.path.join(output_dir, "description.md"), "w", encoding="utf-8") as f:
    f.write(description)

In [2]:
import numpy as np

X_train = np.load("Data/processed_data/X_train.npy")
X_val = np.load("Data/processed_data/X_val.npy")
X_test = np.load("Data/processed_data/X_test.npy")
y_train = np.load("Data/processed_data/y_train.npy")
y_val = np.load("Data/processed_data/y_val.npy")
y_test = np.load("Data/processed_data/y_test.npy")

FileNotFoundError: [Errno 2] No such file or directory: 'Data/processed_data/X_train.npy'

In [ ]:
import keras
from keras import layers, models

inputs = layers.Input(shape=(60,))

x = layers.Dense(128, activation='relu', name='dense0')(inputs)
x = layers.Dense(64, activation='relu', name='dense1')(x)
x = layers.Dense(32, activation='relu', name='dense2')(x)
x = layers.Dense(16, activation='relu', name='dense3')(x)

outputs = layers.Dense(3, activation='softmax')(x)

model = keras.Model(inputs=inputs, outputs=outputs)

model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 60)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense0 (Dense)                  │ (None, 128)            │         7,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense1 (Dense)                  │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense2 (Dense)                  │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense3 (Dense)                  │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 3)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 18,723 (73.14 KB)

 Trainable params: 18,723 (73.14 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
opt = keras.optimizers.Adam(learning_rate=1e-3, amsgrad=True)
model.compile(
    optimizer=opt,
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=8, restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor='val_accuracy', factor=0.5, patience=2, min_lr=1e-7, verbose=1),
]

In [ ]:
# In original project: Epochs = 128, batch_size = 1000
model.fit(X_train, y_train, validation_data=(X_val, y_val), callbacks=callbacks, epochs=100, batch_size=2048)

Epoch 1/100
1614/1614 ━━━━━━━━━━━━━━━━━━━━ 16s 8ms/step - accuracy: 0.7464 - loss: 0.5774 - val_accuracy: 0.7640 - val_loss: 0.5356 - learning_rate: 0.0010
Epoch 2/100
1614/1614 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - accuracy: 0.7676 - loss: 0.5274 - val_accuracy: 0.7705 - val_loss: 0.5211 - learning_rate: 0.0010
Epoch 3/100
1614/1614 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - accuracy: 0.7727 - loss: 0.5172 - val_accuracy: 0.7727 - val_loss: 0.5167 - learning_rate: 0.0010
Epoch 4/100
1614/1614 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - accuracy: 0.7758 - loss: 0.5109 - val_accuracy: 0.7766 - val_loss: 0.5104 - learning_rate: 0.0010
Epoch 5/100
1614/1614 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - accuracy: 0.7779 - loss: 0.5065 - val_accuracy: 0.7773 - val_loss: 0.5072 - learning_rate: 0.0010
Epoch 6/100
1614/1614 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - accuracy: 0.7795 - loss: 0.5034 - val_accuracy: 0.7773 - val_loss: 0.5051 - learning_rate: 0.0010
Epoch 7/100
1612/1614 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy:

In [ ]:
score = model.evaluate(X_test, y_test, verbose=0)
print("Test loss:", score[0])
print("Test accuracy:", score[1])
test_acc = score[1]

model_name=f"model_{model_revision}_acc={test_acc:.4f}.keras"
model.save(os.path.join(model_dir, model_name))

Test loss: 0.4991455078125
Test accuracy: 0.7811318039894104
